# 情報数学Ⅲ 第12回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

In [ ]:
# 使用するライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf # モデルの記述を文字列の数列（formmula）の形で行える
import statsmodels.api as sm 
from scipy import stats

# 日本語表示: Colab環境用
#!pip install japanize_matplotlib
#import japanize_matplotlib

# 日本語表示: 教員確認用（Mac）
# 日本語フォントを指定（ヒラギノ角ゴがmacOSに標準搭載）
plt.rcParams['font.family'] = 'Hiragino Sans'


## 例題1：決定係数を求めてみよう

年齢と年収額の例（以前のデータと異なるので注意）がある．このデータに基づいて回帰分析を行い，決定係数を求めてみよう．

* 決定係数は，0.473（47.3%）となる
* 年収額の散らばりの約半分は年齢によって説明され，残りの半分はそれによって説明できない誤差であると評価できる


In [ ]:
# 年齢と収入のデータを含むリスト
data = [
    {"番号": 1, "年齢": 30, "年収": 400},
    {"番号": 2, "年齢": 33, "年収": 550},
    {"番号": 3, "年齢": 36, "年収": 470},
    {"番号": 4, "年齢": 39, "年収": 620},
    {"番号": 5, "年齢": 42, "年収": 650},
    {"番号": 6, "年齢": 45, "年収": 640},
    {"番号": 7, "年齢": 48, "年収": 680},
    {"番号": 8, "年齢": 51, "年収": 660},
    {"番号": 9, "年齢": 54, "年収": 700},
    {"番号": 10, "年齢": 57, "年収": 640},
    {"番号": 11, "年齢": 60, "年収": 590}
]

### データフレームにする


In [ ]:
df = pd.DataFrame(data)
df.head()

### statsmodelsを使って決定係数を求める

In [ ]:
# モデル化
lm_model = smf.ols(formula='年収 ~ 年齢', 
                   data=df).fit()
# 概要の表示　→　決定係数は表示される
lm_model.summary()

## モデルから取り出すときはこちら


In [ ]:
round(lm_model.rsquared, 3)

## 練習のため自力でも計算してみる

In [ ]:
y = df['年収']             # 応答変数y
y_bar = np.mean(y)         # yの平均値
y_hat = lm_model.predict() # yの当てはめ値

round(np.sum((y_hat - y_bar)**2) / np.sum((y - y_bar)**2), 3)

### TSS：全平方和

In [ ]:
tss = np.sum((y - y_bar)**2)
round(tss, 3)

### ESS：回帰平方和

In [ ]:
ess = np.sum((y_hat - y_bar)**2)
round(ess, 3)

### RSS：残差平方和

In [ ]:
# 残差の取得
e = lm_model.resid
rss = np.sum(e**2)
round(rss, 3)


### TSS = RSS + ESS
* 注意：tss == rss + essだと誤差でFalseになる

In [ ]:
assert np.isclose(tss, rss + ess)
print(np.isclose(tss, rss + ess))

### 決定係数


In [ ]:
# 決定係数を求める
r_squared = ess / tss
round(r_squared, 3)

### 自由度調整済み決定係数（次回に解説予定）

In [ ]:
n = len(df) # サンプルサイズ
d = 1              # 説明変数の数
r2_adj = 1 - ((np.sum(e**2) / (n - d - 1)) / 
    (np.sum((y - y_bar)**2) / (n - 1)))
round(r2_adj, 3)

## 例：分散分析の計算

In [ ]:
data = [
    {"肥料": "なし", "収量": 1},
    {"肥料": "なし", "収量": 3},
    {"肥料": "あり（小）", "収量": 8},
    {"肥料": "あり（小）", "収量": 6},
    {"肥料": "あり（大）", "収量": 10},
    {"肥料": "あり（大）", "収量": 14}
]

### まずは一発で求めるやり方（statsmodels）

In [ ]:
# データフレームの作成
df = pd.DataFrame(data)

# 正規線形モデルの構築（Cはカテゴリ変数として扱うという意味）
anova_model  = smf.ols('収量 ~ C(肥料)', data=df).fit()

# 分散分析の結果
print("=== 一元配置分散分析 ===")
print(sm.stats.anova_lm(anova_model, typ=2))

### 計算過程を理解する(教科書のサンプルコードを参考に作成)

In [ ]:
# 全体の収量配列
y = df["収量"].to_numpy()

# 総平均
y_bar = np.mean(y)

# 各群の平均（順序保証のためカテゴリ順指定）
df["肥料"] = pd.Categorical(df["肥料"], categories=["なし", "あり（小）", "あり（大）"], ordered=True)
y_bar_j = df.groupby("肥料", observed=True).mean()

# 各水準のサンプルサイズ（今回は 2 と仮定）
n_j = 2

# 群の平均を繰り返して effect ベクトルを作成
effect = np.repeat(y_bar_j["収量"], n_j).to_numpy()

# 残差ベクトル
resid = y - effect

# 群間平方和（SSB）
ss_b = np.sum((effect - y_bar) ** 2)

# 群内平方和（SSW）
ss_w = np.sum(resid ** 2)

# 自由度
df_b = len(y_bar_j) - 1
df_w = len(y) - len(y_bar_j)

# 平均平方
ms_b = ss_b / df_b
ms_w = ss_w / df_w

# F比と p 値
f_ratio = ms_b / ms_w
p_value = 1 - stats.f.cdf(f_ratio, df_b, df_w)

# 分散分析表
anova_table_dict = [
    {
        "要因": "群間",
        "平方和": ss_b,
        "自由度": df_b,
        "平均平方": ms_b,
        "F値": f_ratio,
        "p値": p_value
    },
    {
        "要因": "群内",
        "平方和": ss_w,
        "自由度": df_w,
        "平均平方": ms_w,
        "F値": None,
        "p値": None
    },
    {
        "要因": "全体",
        "平方和": ss_b + ss_w,
        "自由度": df_b + df_w,
        "平均平方": None,
        "F値": None,
        "p値": None
    }
]

# データフレーム化して表示
anova_df = pd.DataFrame(anova_table_dict)
print("=== 分散分析表===")
print(anova_df.round(3))


## 例： 二元配置分散分析


In [ ]:
data = [
    {"AM菌": "接種", "濃度": 0,  "値": 16.3},
    {"AM菌": "接種", "濃度": 10, "値": 39.6},
    {"AM菌": "接種", "濃度": 30, "値": 47.7},
    {"AM菌": "接種", "濃度": 50, "値": 48.5},

    {"AM菌": "接種", "濃度": 0,  "値": 18.9},
    {"AM菌": "接種", "濃度": 10, "値": 41.6},
    {"AM菌": "接種", "濃度": 30, "値": 48.0},
    {"AM菌": "接種", "濃度": 50, "値": 51.7},

    {"AM菌": "接種", "濃度": 0,  "値": 18.8},
    {"AM菌": "接種", "濃度": 10, "値": 39.2},
    {"AM菌": "接種", "濃度": 30, "値": 51.0},
    {"AM菌": "接種", "濃度": 50, "値": 46.5},

    {"AM菌": "接種", "濃度": 0,  "値": 16.4},
    {"AM菌": "接種", "濃度": 10, "値": 40.4},
    {"AM菌": "接種", "濃度": 30, "値": 43.7},
    {"AM菌": "接種", "濃度": 50, "値": 43.8},

    {"AM菌": "非接種", "濃度": 0,  "値": 13.6},
    {"AM菌": "非接種", "濃度": 10, "値": 38.3},
    {"AM菌": "非接種", "濃度": 30, "値": 41.9},
    {"AM菌": "非接種", "濃度": 50, "値": 44.1},

    {"AM菌": "非接種", "濃度": 0,  "値": 8.8},
    {"AM菌": "非接種", "濃度": 10, "値": 37.5},
    {"AM菌": "非接種", "濃度": 30, "値": 45.4},
    {"AM菌": "非接種", "濃度": 50, "値": 45.9},

    {"AM菌": "非接種", "濃度": 0,  "値": 17.9},
    {"AM菌": "非接種", "濃度": 10, "値": 36.3},
    {"AM菌": "非接種", "濃度": 30, "値": 45.3},
    {"AM菌": "非接種", "濃度": 50, "値": 44.4},

    {"AM菌": "非接種", "濃度": 0,  "値": 10.8},
    {"AM菌": "非接種", "濃度": 10, "値": 39.0},
    {"AM菌": "非接種", "濃度": 30, "値": 43.7},
    {"AM菌": "非接種", "濃度": 50, "値": 40.7}
]

### 分析の実施

In [ ]:
# データフレームにする
df= pd.DataFrame(data)

# 2元配置分散分析（交互作用あり）
anova_model = smf.ols(formula='値 ~ C(AM菌) * C(濃度)', data=df).fit()

# 分析結果を表示
print("===== 2元配置分散分析（交互作用あり） =====")
print(sm.stats.anova_lm(anova_model, typ=2))

## 演習

このデータは，参考書の著者が実際に実施した，歯の本数と食事を美味しいと感じるかに関する，アンケート調査（一部）である．このデータから歯の本数によって，食事のおいしさの感じ方に差があるかを分散分析を実施せよ．歯の本数によって，食事のおいしさに感じ方の差は（統計的に有意に）あったか？

In [ ]:
data = [
    {"評価": "とてもおいしい", "値": 28},
    {"評価": "とてもおいしい", "値": 27},
    {"評価": "とてもおいしい", "値": 28},
    {"評価": "とてもおいしい", "値": 28},
    {"評価": "とてもおいしい", "値": 32},
    {"評価": "とてもおいしい", "値": 28},
    {"評価": "とてもおいしい", "値": 32},
    {"評価": "とてもおいしい", "値": 28},
    {"評価": "とてもおいしい", "値": 27},
    {"評価": "とてもおいしい", "値": 27},
    {"評価": "とてもおいしい", "値": 28},
    {"評価": "とてもおいしい", "値": 28},
    {"評価": "とてもおいしい", "値": 30},

    {"評価": "おいしい", "値": 29},
    {"評価": "おいしい", "値": 21},
    {"評価": "おいしい", "値": 30},
    {"評価": "おいしい", "値": 28},
    {"評価": "おいしい", "値": 28},
    {"評価": "おいしい", "値": 20},
    {"評価": "おいしい", "値": 31},
    {"評価": "おいしい", "値": 30},
    {"評価": "おいしい", "値": 28},
    {"評価": "おいしい", "値": 17},
    {"評価": "おいしい", "値": 29},
    {"評価": "おいしい", "値": 31},
    {"評価": "おいしい", "値": 32},

    {"評価": "あまりおいしくない", "値": 26},
    {"評価": "あまりおいしくない", "値": 32},
    {"評価": "あまりおいしくない", "値": 20},
    {"評価": "あまりおいしくない", "値": 18},
    {"評価": "あまりおいしくない", "値": 30},
    {"評価": "あまりおいしくない", "値": 11},
    {"評価": "あまりおいしくない", "値": 30},
    {"評価": "あまりおいしくない", "値": 15},
    {"評価": "あまりおいしくない", "値": 22},
    {"評価": "あまりおいしくない", "値": 32},
    {"評価": "あまりおいしくない", "値": 30},
    {"評価": "あまりおいしくない", "値": 25},
    {"評価": "あまりおいしくない", "値": 10},

    {"評価": "おいしくない", "値": 10},
    {"評価": "おいしくない", "値": 18},
    {"評価": "おいしくない", "値": 3},
]


### 分散分析を実施せよ

### 歯の本数によって，食事のおいしさに感じ方の差は（統計的に有意に）あったか？